I tried the following approaches for **Named Entity Recognition**:
1. Traditional / Statistical Models: spaCy
2. Transformer-based Models: BERT, RoBERTa, DeBERTa
3. LLM / LangChain-based NER (Prompt or Retrieval Augmented)

Finally winner between spacy, flair, "dslim/bert-base-NER", and llama3.2:3b (running locally), is **Llama3.2:3b**


In [ ]:
input_text = "I applied for a Software Engineer position at Google, and my application status is 'Interview Scheduled'."


# 1. Traditional Approches

In [20]:
import spacy
ner = spacy.load("en_core_web_sm")
doc = ner(input_text)
entities = {
    "companies": list(set([ent.text for ent in doc.ents if ent.label_ == "ORG"])),
    "jobs": list(
        set(
            [
                ent.text
                for ent in doc.ents
                if ent.label_
                in {"JOB", "TITLE", "WORK_OF_ART", "PRODUCT", "PERSON"}
            ]
        )
    ),
}
entities

{'companies': ['Software', 'Google'], 'jobs': []}

# Flair 

Flair’s pretrained NER models only emit generic labels (PER/ORG/LOC/MISC or OntoNotes tags). They won’t produce custom fields like title or status unless you fine-tune on your own labels or post-process with rules/LLMs.

In [44]:
from flair.data import Sentence
from flair.data import Sentence
from flair.models import SequenceTagger

tagger = SequenceTagger.load("ner-fast")

text = "I applied for a Software Engineer position at Google in Mountain View."
sent = Sentence(text)
tagger.predict(sent)

for ent in sent.get_spans('ner'):
    print(ent.text, ent.get_label('ner').value, ent.score)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/244M [00:00<?, ?B/s]

c:\ProgramData\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mamma\.flair\models\ner-english-fast\models--flair--ner-english-fast. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


2025-10-19 21:51:41,403 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-ORG, S-MISC, B-PER, E-PER, S-LOC, B-ORG, E-ORG, I-PER, S-PER, B-MISC, I-MISC, E-MISC, I-ORG, B-LOC, E-LOC, I-LOC, <START>, <STOP>
Google ORG 0.9966920614242554
Mountain View LOC 0.790443629026413


# 2. Transformer-based Models

In [32]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

nlp = pipeline("ner", model="dslim/bert-base-NER")
res = nlp(input_text)
{item["entity"]:item["word"] for item in res}

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
c:\ProgramData\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


{'B-MISC': 'Software', 'I-MISC': 'Engineer', 'B-ORG': 'Google'}

In [34]:
nlp = pipeline("ner", model="microsoft/deberta-base-mnli")
res = nlp(input_text)
{item["entity"]:item["word"] for item in res}

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

c:\ProgramData\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mamma\.cache\huggingface\hub\models--microsoft--deberta-base-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP downlo

pytorch_model.bin:   0%|          | 0.00/557M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'NEUTRAL': "Ġ'", 'CONTRADICTION': 'Ġapplication', 'ENTAILMENT': "'."}

In [ ]:

classifier = pipeline(
    task="text-classification",
    model="microsoft/deberta-base-mnli",
    device=0,
)

classifier({
    "text": "A soccer game with multiple people playing.",
    "text_pair": "Some people are playing a sport."
})

Some weights of the model checkpoint at microsoft/deberta-base-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


{'label': 'ENTAILMENT', 'score': 0.9938457608222961}

In [41]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "microsoft/deberta-base-mnli"
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base-mnli")
model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-base-mnli", device_map="auto")

inputs = tokenizer(
    "A soccer game with multiple people playing.",
    "Some people are playing a sport.",
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    logits = model(**inputs).logits
    print(logits)
    predicted_class = logits.argmax().item()

labels = ["contradiction", "neutral", "entailment"]
print(f"The predicted relation is: {labels[predicted_class]}")

Some weights of the model checkpoint at microsoft/deberta-base-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tensor([[-3.9121, -0.5010,  4.6159]])
The predicted relation is: entailment


# 3. LLM Based Extraction

In [16]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel
from langchain_ollama import ChatOllama
llm = ChatOllama(
        model="llama3.2:3b",
        base_url="http://localhost:11434",
        validate_model_on_init=True,
        request_timeout=60.0
    )
print("Model 'llama3.2:3b' is running and ready.", llm.invoke("what is ollama in 3 words"))
class JobEntity(BaseModel):
    company: str
    title: str
    status: str

parser = PydanticOutputParser(pydantic_object=JobEntity)
prompt = PromptTemplate(
    template="Extract company, title, and application status from the text: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm | parser

input_text = "I applied for a Software Engineer position at Google, and my application status is 'Interview Scheduled'."

chain.invoke({"text": input_text})

Model 'llama3.2:3b' is running and ready. content='Japanese street food.' additional_kwargs={} response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-10-20T04:13:15.7242669Z', 'done': True, 'done_reason': 'stop', 'total_duration': 376511600, 'load_duration': 68149900, 'prompt_eval_count': 34, 'prompt_eval_duration': 286877400, 'eval_count': 5, 'eval_duration': 19912600, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'} id='lc_run--a9d35ffd-fe0b-4bc5-a333-ccfc97288b22-0' usage_metadata={'input_tokens': 34, 'output_tokens': 5, 'total_tokens': 39}


JobEntity(company='Google', title='Software Engineer', status='Interview Scheduled')